In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import shutil
import tempfile
import pandas as pd
import torch
from torch.nn import MSELoss
from monai.apps import download_url, download_and_extract
from monai.config import print_config
from monai.data import DataLoader, Dataset, CacheDataset

from monai.transforms import (
    Compose,
    LoadImaged,
    RandAffined,
    Resized,
    ScaleIntensityRanged,
    CropForegroundd,
    RandRotated,
)
from monai.utils import set_determinism, first
import glob
import torch
import torch.nn as nn


In [ ]:
set_determinism(seed=2025)

In [ ]:
root_dir = "./"
data_dir = "./"
train_images1 = sorted(glob.glob(os.path.join(data_dir, "3T_train", "*1rorg.nii.gz")))
train_images2 = sorted(glob.glob(os.path.join(data_dir, "3T_train", "*2rorg.nii.gz")))
train_images3 = sorted(glob.glob(os.path.join(data_dir, "3T_train", "*3rorg.nii.gz")))
labels_train = pd.read_csv('./B200/train_labels.csv')
train_label_array = np.array(labels_train['label'])
train_files = [{"image1": image_name1, "image2": image_name2, "image3": image_name3, "label": train_label_array[i]}
              for i, (image_name1, image_name2, image_name3) in enumerate(zip(train_images1, train_images2, train_images3))]

In [ ]:

data_dir_val = "./B101"

val_images1 = sorted(glob.glob(os.path.join(data_dir_val, "3T_Val", "*1rorg.nii.gz")))

val_images2 = sorted(glob.glob(os.path.join(data_dir_val, "3T_Val", "*2rorg.nii.gz")))

val_images3 = sorted(glob.glob(os.path.join(data_dir_val, "3T_Val", "*3rorg.nii.gz")))

labels_val = pd.read_csv('./B101/.csv')
val_label_array  = np.array(labels_val['label'])

file_ids = [os.path.basename(f).split('3rorg.nii.gz')[0] for f in val_images3]

val_files = [
    {
        "image1": image_name1,
        "image2": image_name2,
        "image3": image_name3,
        "label": val_label_array[i],
        "file_id": file_ids[i] 
    }
    for i, (image_name1, image_name2, image_name3) in enumerate(zip(val_images1, val_images2, val_images3))
]

In [ ]:
data_dir_val_external = "./B101"

val_external_images1 = sorted(glob.glob(os.path.join(data_dir_val_external, "3T_Val_external_3_12", "*1rorg.nii.gz")))

val_external_images2 = sorted(glob.glob(os.path.join(data_dir_val_external, "3T_Val_external_3_12", "*2rorg.nii.gz")))

val_external_images3 = sorted(glob.glob(os.path.join(data_dir_val_external, "3T_Val_external_3_12", "*3rorg.nii.gz")))

labels_external_val = pd.read_csv('./Val_label_external.csv')
val_external_label_array  = np.array(labels_external_val['label'])

file_ids_ex = [os.path.basename(f).split('3rorg.nii.gz')[0] for f in val_external_images3]

val_external_files = [{"image1": image_name1,
                        "image2": image_name2, 
                        "image3": image_name3, 
                        "label": val_external_label_array[i],
                        "file_id": file_ids_ex[i]} 
              for i, (image_name1, image_name2, image_name3) in enumerate(zip(val_external_images1, val_external_images2, val_external_images3))]


In [ ]:
train_transforms = Compose(
    [
        LoadImaged(keys=["image1", "image2", "image3"], ensure_channel_first=True),
        ScaleIntensityRanged(
            keys=["image1","image2","image3"],
            a_min=0,
            a_max=250,
            b_min=0,######
            b_max=250,
            clip=True,
        ),
        RandRotated(keys=["image1","image2","image3"], prob=0.5, range_x=10.0),
        RandAffined(keys=["image1","image2","image3"], prob=0.5, translate_range=10,rotate_range=(np.pi / 36, np.pi / 36, np.pi / 4),scale_range=(0.15, 0.15, 0.15)),
        Resized(
            keys=["image1","image2", "image3"],
            mode=("trilinear"),#####
            align_corners=(True),#####
            spatial_size=(96, 96, 14),
        ),
    ]
)
val_transforms = Compose(
    [
        LoadImaged(keys=["image1", "image2", "image3"], ensure_channel_first=True),
        ScaleIntensityRanged(
            keys=["image1","image2","image3"],
            a_min=0,
            a_max=250,
            b_min=0,######
            b_max=250,
            clip=True,
        ),
        Resized(
            keys=["image1","image2", "image3"],
            mode=("trilinear"),#####
            align_corners=(True),
            spatial_size=(96, 96, 14),
        ),
    ]
)
val_external_transforms = Compose(
    [
        LoadImaged(keys=["image1", "image2", "image3"], ensure_channel_first=True),
        ScaleIntensityRanged(
            keys=["image1","image2","image3"],
            a_min=0,
            a_max=250,
            b_min=0,######
            b_max=250,
            clip=True,
        ),
        Resized(
            keys=["image1","image2", "image3"],
            mode=("trilinear"),#####
            align_corners=(True),
            spatial_size=(96, 96, 14),
        ),
    ]
)

In [ ]:
train_ds = CacheDataset(data=train_files, transform=train_transforms, cache_rate=1.0, num_workers=4)
train_loader = DataLoader(train_ds, batch_size=253, shuffle=True, num_workers=1)
val_ds = CacheDataset(data=val_files, transform=val_transforms, cache_rate=1.0, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=1)
val_external_ds = CacheDataset(data=val_external_files, transform=val_external_transforms, cache_rate=1.0, num_workers=4)
val_external_loader = DataLoader(val_external_ds, batch_size=1, num_workers=1)

In [ ]:
from monai.networks.nets import SEResNet50
import torch.nn as nn
device = torch.device("cuda:0")
import torch.nn.functional as F
class SEResNet50FeatureExtractor(SEResNet50):
    def __init__(self, spatial_dims, in_channels, num_classes):
        super(SEResNet50FeatureExtractor, self).__init__(spatial_dims=spatial_dims, in_channels=in_channels, num_classes=num_classes)

    def forward(self, x):
    
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x) 
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.adaptive_avg_pool(x)
        features = torch.flatten(x, 1)
        x = self.last_linear(features)
        return features, x
    
model_N = SEResNet50FeatureExtractor(spatial_dims=3, in_channels=1, num_classes=2)
model_A = SEResNet50FeatureExtractor(spatial_dims=3, in_channels=1, num_classes=2)
model_V = SEResNet50FeatureExtractor(spatial_dims=3, in_channels=1, num_classes=2)
checkpoint_N = torch.load('./best_metric_model_N.pth')
state_dict_N = checkpoint_N
state_dict_N = {k.replace('module.', ''): v for k, v in state_dict_N.items()}
checkpoint_A = torch.load('./best_metric_model_A.pth')
state_dict_A = checkpoint_A
state_dict_A = {k.replace('module.', ''): v for k, v in state_dict_A.items()}
checkpoint_V = torch.load('./best_metric_model_V.pth')
state_dict_V = checkpoint_V
state_dict_V = {k.replace('module.', ''): v for k, v in state_dict_V.items()}

model_N.load_state_dict(state_dict_N, strict=True)
model_A.load_state_dict(state_dict_A, strict=True)
model_V.load_state_dict(state_dict_V, strict=True)



In [ ]:
from perceiver_pytorch.multi_modality_perceiver import MultiModalityPerceiver, InputModality
U_modality = InputModality(
    name='U',
    input_channels=2048,  # number of channels for each token of the input
    input_axis=1,  # number of axes, 2 for images
    num_freq_bands=3,  # number of freq bands, with original value (2 * K + 1)
    max_freq=1.,  # maximum frequency, hyperparameter depending on how fine the data is
)

C_modality = InputModality(
    name='C',
    input_channels=2048,  # number of channels for each token of the input
    input_axis=1,  # number of axes, 2 for images
    num_freq_bands=3,  # number of freq bands, with original value (2 * K + 1)
    max_freq=1.,  # maximum frequency, hyperparameter depending on how fine the data is
)

N_modality = InputModality(
    name='N',
    input_channels=2048,  # number of channels for each token of the input
    input_axis=1,  # number of axes, 2 for images
    num_freq_bands=3,  # number of freq bands, with original value (2 * K + 1)
    max_freq=1.,  # maximum frequency, hyperparameter depending on how fine the data is
)
perceiver_multi = MultiModalityPerceiver(
    modalities=(U_modality, C_modality, N_modality),
    depth=1,  # depth of net, combined with num_latent_blocks_per_layer to produce full Perceiver
    num_latents=256,
    # number of latents, or induced set points, or centroids. different papers giving it different names
    latent_dim=512,  # latent dimension
    cross_heads=1,  # number of heads for cross attention. paper said 1
    latent_heads=8,  # number of heads for latent self attention, 8
    cross_dim_head=64,
    latent_dim_head=64,
    num_classes=2,  # output number of classes
    attn_dropout=0.1,
    ff_dropout=0.6,
    weight_tie_layers=True,
    num_latent_blocks_per_layer=3  # Note that this parameter is 1 in the original Lucidrain implementation
)


In [ ]:
class multi_perceiver(nn.Module):
    def __init__(self):
        super(multi_perceiver, self).__init__()
        self.model_N = model_N 
        self.model_A = model_A
        self.model_V = model_V
        self.perceiver_multi = perceiver_multi
    def forward(self, image_N, image_A, image_V):
        with torch.no_grad():
            x_N = self.model_N(image_N)[0]
            x_A = self.model_A(image_A)[0]
            x_V = self.model_V(image_V)[0]
            # print(x_N.shape),print(x_A.shape),print(x_V.shape)
            x_N = x_N.unsqueeze(2).permute(0, 2, 1).type(torch.FloatTensor).to(device)
            x_A = x_A.unsqueeze(2).permute(0, 2, 1).type(torch.FloatTensor).to(device)
            x_V = x_V.unsqueeze(2).permute(0, 2, 1).type(torch.FloatTensor).to(device)
        # print(x_N.shape),print(x_A.shape),print(x_V.shape)
        pred = self.perceiver_multi({'U': x_N,
                                     'C': x_A,
                                     'N': x_V})          
        return pred
model = multi_perceiver().to(device)

In [ ]:
device = torch.device("cuda:0")
model = nn.DataParallel(model, device_ids=[0])
model = model.to(device)

In [ ]:
import torch
import numpy as np
labels = [train_ds[i]['label'] for i in range(len(train_ds))]
counts = np.bincount(labels)
freqs = counts / np.sum(counts)
print(freqs)
class_weights = 1 / np.log(1.02 + freqs)
print(class_weights)
class_weights_tensor = torch.from_numpy(class_weights.astype(np.float32)).to(device)

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau
loss_function = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

from process import EarlyStopping
optimizer = torch.optim.Adam(model.parameters(), 1e-3)
patience = 20
early_stopping = EarlyStopping(patience=patience, verbose=True,path='./checkpoint_perceiver.pt')
import pandas as pd
import sklearn.metrics as metrics
eval_df = pd.DataFrame(columns=['epoch', 'accuracy', 'auc', 'specificity', 'sensitivity'])
metric_count = 0
num_correct = 0
y_true = []
y_pred = []
best_metric_model_path = './best_metric_model_gru.pth'

val_interval = 2
best_metric = -1    
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
writer = SummaryWriter()
max_epochs = 100
red_lr = ReduceLROnPlateau(optimizer, patience=5, verbose=1, factor=0.8)
valid_losses = []
avg_valid_losses = [] 
for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        optimizer.zero_grad()
        image1 = batch_data["image1"].to(device)
        image2 = batch_data["image2"].to(device)
        image3 = batch_data["image3"].to(device)
        label = batch_data["label"].to(device)
        output = model(image1,image2,image3)
        loss = loss_function(output, label)   

    

        loss.backward()


        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        
        num_correct = 0.0
        metric_count = 0
        for val_data in val_loader:
            with torch.no_grad():
                val_loss = 0.0
                val_image1 = val_data["image1"].to(device)
                val_image2 = val_data["image2"].to(device)
                val_image3 = val_data["image3"].to(device)
                val_label = val_data["label"].to(device)
                val_output = model(val_image1,val_image2,val_image3)
                val_loss = loss_function(val_output, val_label)
                valid_losses.append(val_loss.item())
                metric_count += len(val_label)
                num_correct += (val_output.argmax(dim=1) == val_label).sum().item()
                y_true.extend(val_label.cpu().numpy())
                y_pred.extend(val_output.argmax(dim=1).cpu().numpy())
        valid_loss = np.average(valid_losses)
        avg_valid_losses.append(valid_loss)
        print_msg = (f'[{epoch + 1}/{max_epochs}] ' +
                     f'train_loss: {epoch_loss:.5f} ' +
                     f'valid_loss: {valid_loss:.5f}')
        print(print_msg)
        valid_losses = []
        early_stopping(valid_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break
        accuracy = num_correct / metric_count
        fpr, tpr, _ = metrics.roc_curve(y_true, y_pred, pos_label=1)
        auc = metrics.auc(fpr, tpr)
        tn, fp, fn, tp = metrics.confusion_matrix(y_true, y_pred).ravel()
        specificity = tn / (tn + fp)
        sensitivity = tp / (tp + fn)
        new_row = {'epoch': epoch + 1,
                   'accuracy': accuracy,
                   'auc': auc,
                   'specificity': specificity,
                   'sensitivity': sensitivity}
        eval_df = pd.concat([eval_df, pd.DataFrame(new_row, index=[0])], ignore_index=True)      
        metric = auc
        red_lr.step(auc) 
        print(f"Current epoch: {epoch+1} current auc: {metric:.4f} ")
        metric_values.append(metric)
        if metric > best_metric:
            best_metric = metric
            best_metric_epoch = epoch + 1
            torch.save(model.state_dict(), best_metric_model_path)
            print("saved new best metric model")
            print(f"Best auc: {best_metric:.4f} at epoch {best_metric_epoch}")
            writer.add_scalar("val_accuracy", metric, epoch + 1)
eval_df.to_csv(f'./checkpoint_perceiver/eval_metrics_epoch_{epoch+1}.csv', index=False)
print(f"Training completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()